# Manual L1 annotation

Use this notebook to review per-cluster markers for the L1 Leiden clusters from
`results/05_annotation/adata_annotated.h5ad`, look at spatial maps per cluster,
and write a `cluster_labels.yaml` that the pipeline picks up when run with
`annotation.method: manual`.

Workflow:
1. Run the pipeline through stage 3 with `annotation.method: markers` (auto first pass).
2. Open this notebook, inspect markers and spatial maps, and edit the labels dict.
3. Save `cluster_labels.yaml` next to this notebook.
4. Set `annotation.method: manual` and `annotation.manual_labels_l1: <path>` in `config/config.yaml`.
5. Re-run `visium-brain cluster` to apply the manual labels and re-derive L2.

In [ ]:
from pathlib import Path
import yaml
import scanpy as sc
from visium_brain.io import read_h5ad
from visium_brain.utils import load_config, output_paths

cfg = load_config('../config/config.yaml')
paths = output_paths(cfg)
adata = read_h5ad(paths['annotation'] / 'adata_annotated.h5ad')
adata

In [ ]:
# Top markers per L1 Leiden cluster.
# use_raw=True so the log-fold-changes come from the log1p snapshot in adata.raw
# rather than the scaled values left in .X by sc.pp.scale.
cluster_key = 'leiden_l1' if 'leiden_l1' in adata.obs else 'leiden'
sc.tl.rank_genes_groups(adata, groupby=cluster_key, method='wilcoxon', use_raw=True)
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
# Per-cluster mean of the auto marker-score panels (written by the pipeline)
import pandas as pd
scores = adata.uns.get('cluster_marker_scores')
pd.DataFrame(scores) if scores is not None else None

In [ ]:
sc.pl.umap(adata, color=[cluster_key, 'condition', 'sample_id'], ncols=2)

In [ ]:
# Spatial map per sample, coloured by Leiden cluster
for sid in adata.obs['sample_id'].cat.categories:
    sub = adata[adata.obs['sample_id'] == sid].copy()
    sub.uns['spatial'] = {sid: adata.uns['spatial'][sid]}
    sc.pl.spatial(sub, color=cluster_key, library_id=sid, spot_size=cfg['plotting']['spot_size'])

In [ ]:
# Edit this mapping with the cell-type calls you derived from the markers above,
# then save to cluster_labels.yaml. Re-run the pipeline with annotation.method=manual.
labels = {str(c): 'TBD' for c in adata.obs[cluster_key].cat.categories}
labels

In [ ]:
out = Path('cluster_labels.yaml')
with open(out, 'w') as fh:
    yaml.safe_dump({'cluster_key': cluster_key, 'labels': labels}, fh, sort_keys=False)
print('Wrote', out.resolve())